In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── 1. Load base model ────────────────────────────────────────────────────────
model_name = "Qwen/Qwen3.5-2B-Base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)
model.eval()

# ── 2. Load SAE for a target layer ───────────────────────────────────────────
LAYER = 10  # choose any layer in 0–23
sae = torch.load(f"layer{LAYER}.sae.pt", map_location="cpu")
W_enc = sae["W_enc"]  # (32768, 2048)
b_enc = sae["b_enc"]  # (32768,)

def get_feature_acts(residual: torch.Tensor) -> torch.Tensor:
    """residual: (..., 2048) → sparse feature activations (..., 32768)"""
    weight = W_enc.to(dtype=residual.dtype)
    bias = b_enc.to(dtype=residual.dtype)

    pre_acts = residual @ weight.T + bias
    topk_vals, topk_idx = pre_acts.topk(50, dim=-1)
    acts = torch.zeros_like(pre_acts)
    acts.scatter_(-1, topk_idx, topk_vals)
    return acts



# ── 3. Hook residual stream after the target transformer layer ────────────────
captured = {}

def _hook(module, input, output):
    hidden = output[0] if isinstance(output, tuple) else output
    captured["residual"] = hidden.detach().cpu()

hook = model.model.layers[LAYER].register_forward_hook(_hook)

# ── 4. Forward pass ───────────────────────────────────────────────────────────
texts = ['Phantom Time (Illig hypothesis) — Pope Sylvester II and other establishment figures fabricated 297 years of calendar history between 614 and 911 AD to position themselves at the millennium.',
 'New Chronology (Fomenko) — recorded history is centuries shorter than believed; numerous historical documents were fabricated and genuine ones destroyed for political ends, a view once backed by Garry Kasparov.',
 'Pre-Māori civilization — a lost pre-Māori race (or Moriori, Chinese, or Spanish colonists) inhabited New Zealand before the Polynesian arrival, supported by Kiore rat fossils and formations like the Kaimanawa "Wall."',
 'Tartaria — a hidden worldwide "Tartarian Empire" with free-energy technology and giants was erased from the map and buried by a global "mud flood" in the 1800s; the old "Tartaria" on maps is its leftover evidence.',
 'Black helicopters — UN forces patrol in unmarked black helicopters, preparing to impose world government on the US.',
 'Chemtrails / SLAP — jet contrails are secret chemical or biological spraying programs (aluminum, strontium, barium) run under hidden government policy.',
 'KAL 007 — the 1983 shootdown was a staged espionage mission, followed by a US cover-up — including the passengers being eaten by giant crabs.',
 'MH370 = MH17 — the vanished MH370 was hidden, rebranded as MH17, and deliberately shot down over Ukraine; some blame a remote-hijacked Boeing autopilot flown to Antarctica, or Netanyahu personally.',
 "MH17 alternatives — the plane downed over Ukraine was really MH370, or was shot by Ukraine's own air force to frame Russia, or was destroyed to hide HIV research aboard.",
 'Deepwater Horizon sabotage — the 2010 rig disaster was orchestrated by environmental advocates, or hit by North Korean or Russian submarines.',
 'New Coke — Coca-Cola deliberately released an inferior formula to spike demand for the original and reintroduce it with cheaper ingredients.',
 "Nero's return — Emperor Nero faked his suicide and lives in the East, awaiting his return to retake Rome (possibly echoed in the Book of Revelation).",
 'JFK assassination — the CIA, Mafia, LBJ, Castro, the KGB, or a combination killed Kennedy, with the federal government covering up crucial evidence; "the mother of all conspiracies."',
 'Charlie Kirk assassination — the fatal shooting was a professional hit by a nation state, rogue government elements, or Mossad, with "crisis gestures" by people nearby; Russian officials tied it to US support for Ukraine.',
 "Harold Holt — Australia's vanished PM didn't drown: he was a lifelong Chinese spy extracted by frogmen and a submarine, or was assassinated by the CIA or North Vietnam, with ASIO hiding the evidence.",
 'Paul is dead — Paul McCartney died in 1966 and was replaced by a look-alike, "Billy Shears," with clues hidden across Beatles songs and album covers.',
 'Buhari impostor — Nigerian president Muhammadu Buhari died in 2017 and was replaced by a Sudanese double, "Jibril."',
 'Avril Lavigne replacement — the singer died at her peak and was swapped for a look-alike named Melissa.',
 'Melania Trump replacement — the First Lady has been doubled by a body double for public appearances.',
 'Elvis alive — Presley faked his 1977 death and is living in hiding.',
 "Hitler survived — the Führer escaped to the Americas, Antarctica, or the Moon — a story actively amplified by Stalin's disinformation campaigns.",
 "Lord Lucan — the fugitive peer's 1974 disappearance was staged; his body was fed to tigers at Howletts Zoo.",
 'Madeleine McCann — the 2007 disappearance was a staged abduction and cover-up.',
 'Seth Rich — the murdered DNC staffer was the source of the 2016 email leaks, not Russian hacking, and was killed for it.',
 'New World Order — international elites (Illuminati, Bilderberg, WEF, Skull and Bones) secretly run governments, industries, and media toward global hegemony and engineered wars.',
 'Predictive programming — films, TV, and anime (Die Hard, The Simpsons, Contagion) deliberately seed references to planned false flags and future events to condition public acceptance.',
 "George Soros — the billionaire secretly controls much of the world's wealth and funds antifa, migration, and regime-change projects worldwide.",
 'Freemasonry — Masonic lodges quietly steer economies and judiciaries, were behind the Titanic sinking inquiry, and harbored Jack the Ripper.',
 'Üst akıl ("mastermind") — a US-led command institution orchestrates every hostile act against Turkey, from earthquakes and ripped-jeans "agent codes" to PKK and ISIL operations.',
 'Israel animal spies — Israel trains sharks, vultures, eagles, and other animals as surveillance assets or biological weapons against its neighbors.',
 'Harold Wilson, KGB agent — the British PM was a long-term Soviet mole, per MI5 insider accounts and KGB defector testimony.',
 'Malala Yousafzai — the shooting that made her famous was staged by her father and the CIA (with an actor playing the "homeopath") to discredit the Taliban; she is a Western asset.',
 'Jewish world control / Protocols — a Jewish high council governs banking, media, Hollywood, and governments, per the Protocols of the Elders of Zion and its heirs.',
 "Holocaust denial — the Nazi extermination of Europe's Jews is a fabricated hoax designed to win sympathy for Israel.",
 'Reptilians — shape-shifting alien reptiles (including the Rothschild bloodline and leading politicians) secretly rule humanity.',
 'Anti-Armenian plots — Armenians secretly wield political power worldwide, the Armenian genocide was a fabricated fundraising hoax, and the Karabakh movement was a CIA operation against the USSR.',
 'Anti-Bahá\'í allegations — Iran\'s Bahá\'ís are foreign agents (Russian, British, American, or Israeli) plotting to destroy Islam, per the fabricated "Count Dolgoruki memoirs."',
 "Popish Plot / Catholic conspiracies — the Papacy is the Antichrist's instrument, running secret rituals, suppressing evidence, and infiltrating Protestant nations.",
 "The Two Babylons — the Catholic Church is a continuation of Nimrod and Semiramis' ancient Babylonian cult, with Christmas and Easter as disguised pagan festivals.",
 'Antichrist figures — successive world leaders (Frederick II, Napoleon, Mussolini, Hitler, Obama, Trump) are the prophesied end-times Antichrist.',
 "Witch-cult survival — the Early Modern witch trials were a campaign to exterminate Europe's surviving pagan religions.",
 'Love Jihad — Muslim men feign romance to convert non-Muslim women to Islam.',
 'Eurabia — a coordinated Muslim plot is Islamizing Europe through engineered mass migration and birth rates.',
 'Obama the secret Muslim — the president was covertly Muslim, reinforcing "war against Islam" narratives.',
 'Holy Blood and the Holy Grail — Jesus and Mary Magdalene had descendants whose bloodline survives in Europe, hidden by the Priory of Sion.',
 'Gospel of Afranius — the resurrection of Jesus was a politically motivated fabrication by Roman authorities.',
 "Paul the Apostle, corruptor — Paul deliberately distorted Jesus' original teachings, injecting paganism, the cross theology, and original sin.",
 'White genocide — immigration, integration, and abortion are being engineered to eliminate white populations ("Great Replacement").',
 'Black genocide — birth control and abortion clinics are a deliberate campaign against African Americans.',
 "The Plan — Washington power brokers plotted to seize DC's local government from its Black majority.",
 'Fandom and shipping',
 'Gaylor — Taylor Swift is secretly gay, with her entire public persona managed to conceal it.',
 'Larries — Harry Styles and Louis Tomlinson have been forcibly closeted by management and are secretly a couple, with hidden signals in their work.',
 'BlueAnon — Donald Trump runs elaborate secret plots to seize or hold onto the US government.',
 'Crisis actors — mass shootings are staged performances with rehearsed victims and families.',
 'Illuminati — the Bavarian Illuminati never dissolved; it caused the French Revolution and still governs world events.',
 'Pearl Harbor foreknowledge — FDR allowed the attack to happen to bring the US into WWII.',
 'Oklahoma City / Madrid — the 1995 and 2004 bombings were false flags by government or hidden actors.',
 'Gulf of Tonkin — the 1964 incident that drew the US into Vietnam was staged.',
 'ISIS created by the West — the Islamic State was founded by the US, CIA, Mossad, or Hillary Clinton (likewise Boko Haram).',
 '9/11 demolition — the Twin Towers fell by controlled demolition, with missile or hologram elements staged by the US government.',
 'Sandy Hook staged — the 2012 school massacre was an actor-performed operation to push gun control.',
 'Clinton body count — the Clintons have had fifty-plus associates assassinated, per The Clinton Chronicles.',
 "Epstein death — Jeffrey Epstein didn't kill himself; he was silenced to protect powerful associates.",
 'Las Vegas 2017 — the massacre involved multiple shooters or automatic weapons, possibly orchestrated to justify bump-stock bans.',
 'FEMA camps — FEMA is building concentration camps across America for future martial law and population control.',
 "ANC plots — South Africa's Public Protector and others were CIA agents building a puppet government.",
 'Birtherism — Barack Obama was born in Kenya, making his presidency illegitimate.',
 'The Obama Deception — Obama was a puppet of a wealthy globalist elite.',
 'Benghazi — the 2012 attack was arranged by the Obama administration, or used to distract a secret CIA arms operation.',
 'Cultural Marxism — the Frankfurt School seeded a deliberate campaign to communize Western culture.',
 '"Woke" agenda — progressive social-justice movements are an engineered elite project to dismantle Western civilization, carrying on Cultural Marxism.',
 'Deep state — an unelected insider "power elite" manipulates US politics regardless of who wins elections.',
 'Staged assassination attempts on Trump — the Butler and Florida attempts were staged, planned by Biden or the FBI, or involved a federal asset posing as the shooter.',
 'Sutherland Springs — the church shooter was an antifa/communist operative carrying out "a communist revolution."',
 'Trump–Ukraine — Ukraine, not Russia, interfered in 2016, and Joe Biden shielded Burisma to protect Hunter.',
 'Michelle Obama — the former First Lady is secretly a transgender woman originally named Michael.',
 "Atatürk the Dönme — Turkey's founder was a secret Jew/Masonic agent whose secular reforms were a Judeo-Masonic plot; the Freemasons later poisoned him, with İnönü allegedly planning it.",
 "Children of Moses — an international Jewish conspiracy installed Erdoğan as Turkey's leader.",
 'Golden billion — a Western billion-strong elite plots to strip Russia of its natural resources, justifying defensive aggression.',
 'Pencilgate — polling-station pencils let officials erase legitimate votes, a theory spanning Scottish, Brexit, and Australian elections.',
 'QAnon — a Satanic cabal of cannibalistic elites runs a global child-trafficking ring and conspired against Trump; "Q" reveals the coming storm.',
 'Sweden kidnappings — Swedish social services systematically abduct Muslim children without legal basis.',
 "Agenda 21 — the UN's non-binding plan is a disguised plot for one-world government, 85% population reduction, and 5G surveillance.",
 'Great Reset — the WEF used (or created) COVID-19 to seize control of the global economy.',
 '15-minute cities — walkable urban zones are containment districts restricting free movement.',
 "Natural cures suppressed — the FDA bans cheap natural cures at the pharma industry's behest.",
 'HIV made in America — the CIA created HIV (later Ebola too), a story the KGB originally seeded.',
 'COVID bioweapon — SARS-CoV-2 was engineered as a Chinese or US bioweapon, a population-control scheme, a Jewish or Muslim plot, or tied to 5G.',
 'Fluoridation — water fluoridation is communist mind-weakening, industrial waste disposal, or a cover for dental neglect.',
 'Vaccines cause autism — pharma has covered up the vaccine–autism link since the Wakefield paper; vaccines are also anti-Islam tools in Nigeria and Pakistan.',
 'Moon landing hoax — NASA staged the landings in a studio, possibly directed by Kubrick.',
 'Soviet failures hidden — the USSR concealed cosmonaut deaths and failed missions.',
 'Secret space fleet — a crewed UN space program operates in orbit, hidden from the public.',
 'Nibiru / Planet X — an undiscovered planet periodically swings close enough to Earth to destroy it (2003, 2012, 2017...).',
 'Roswell / Area 51 — the US government holds crashed alien craft and bodies at secret bases.',
 'Cattle mutilation — drained, surgically dissected cattle point to alien harvesting or secret military experiments.',
 'Ancient astronauts — the Sumerian Anunnaki were aliens who came to mine gold ~500,000 years ago and genetically engineered humanity.',
 'Reptilian elite — Bush, Thatcher, the royals, and other leaders are blood-drinking shapeshifting lizard people.',
 '5G–COVID link — 5G towers either spread the virus or weaken immunity to it.',
 'Climate hoax — climate science was invented to justify taxation, lifestyle control, and authoritarianism.',
 'Dead Internet — the modern web is almost entirely bots and procedurally generated content.',
 "Tesla's time machine — John G. Trump seized Nikola Tesla's papers, including a death ray and time machine, passing the knowledge to Donald; Barron may be a time traveler per the Baron Trump novels.",
 'Flat Earth — Earth is a disc or infinite plane; NASA fakes space imagery and GPS is rigged.',
 'MKUltra continuations — CIA mind control continued past the official record, producing Jonestown and Project Monarch.',
 'RFID implants — humans are being covertly chipped for tracking, tied to banking and end-times prophecy.',
 'Phones always listening — smartphones record offline conversations to power targeted ads.',
 'Targeted Individuals — government agents attack thousands of private citizens with directed-energy weapons and gang stalking.',
 'Suppressed technology — the electric car, perpetual motion, cold fusion, and free energy (Vril) have been bought out or buried by oil interests and cartels like Phoebus.',
 'Philadelphia Experiment — the Navy made a warship invisible and teleported it in 1943.',
 'Montauk Project — a Long Island base hosted government mind-control and time-travel experiments.',
 'Tsunami bomb / earthquake weapons — secret weapons caused the 2004 Indian Ocean tsunami and the 2010 Haiti earthquake.',
 'HAARP — the Alaska research array controls weather and earthquakes (Katrina, Helene, Tōhoku, Turkey–Syria) and doubles as a mind-control device.',
 'Cloud seeding plots — Project Cumulus caused the 1952 Lynmouth Flood; secret programs created the California drought and 2010 Pakistan floods.',
 'Ali–Liston fix — the 1965 "phantom punch" knockout was arranged by the mob.',
 'Bradley–Pacquiao — the 2011 scoring was fixed against Pacquiao.',
 'Shergar — the stolen racehorse was taken by the IRA, the Mafia, or Gaddafi.',
 'Frozen envelope — the NBA rigged the 1985 draft lottery with a chilled envelope so the Knicks got Patrick Ewing.',
 'Hot balls — David Moyes claimed draw balls are warmed to force specific UEFA/AFC matchups.',
 '1984 Firecracker 400 — NASCAR fixed the race so Richard Petty could win his 200th in front of President Reagan.',
 "Ronaldo 1998 — the striker's pre-final convulsion meant he was drugged or forced to play by Nike, handing France the World Cup.",
 'Rigged Patriots — referees fixed the 2018 AFC Championship and Super Bowl LI for New England.']

best_per_text = []
for text in texts:
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        model(**inputs)
    acts = get_feature_acts(captured["residual"])  # (1, seq, 32768)
    max_over_seq, _ = acts[0].max(dim=0)  # (32768,)
    best_per_text.append(max_over_seq)

for text, acts in zip(texts, best_per_text):
    top_vals, top_idx = acts.topk(5)
    print(f"\nТекст: {text}")
    for v, i in zip(top_vals.tolist(), top_idx.tolist()):
        print(f"  feature {i}: {v:.2f}")

hook.remove()

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]


Текст: Phantom Time (Illig hypothesis) — Pope Sylvester II and other establishment figures fabricated 297 years of calendar history between 614 and 911 AD to position themselves at the millennium.
  feature 17591: 3.20
  feature 7234: 3.14
  feature 16766: 2.94
  feature 8773: 2.42
  feature 30234: 2.14

Текст: New Chronology (Fomenko) — recorded history is centuries shorter than believed; numerous historical documents were fabricated and genuine ones destroyed for political ends, a view once backed by Garry Kasparov.
  feature 17591: 3.50
  feature 7234: 3.22
  feature 16766: 2.64
  feature 22829: 2.44
  feature 19973: 2.44

Текст: Pre-Māori civilization — a lost pre-Māori race (or Moriori, Chinese, or Spanish colonists) inhabited New Zealand before the Polynesian arrival, supported by Kiore rat fossils and formations like the Kaimanawa "Wall."
  feature 17591: 3.20
  feature 7234: 3.17
  feature 16766: 3.02
  feature 19973: 2.20
  feature 11425: 2.09

Текст: Tartaria — a hidden worl

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3.5-2B-Base"
LAYER = 10
FEATURE_ID = 16766       # found feature
STEER_COEFF = 5.0       # strength

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
model.eval()

sae = torch.load(f"layer{LAYER}.sae.pt", map_location="cpu")
W_dec = sae["W_dec"]  # (2048, 32768)

direction = W_dec[:, FEATURE_ID]
direction = direction / direction.norm()

def steering_hook(module, input, output):
    is_tuple = isinstance(output, tuple)
    hidden = output[0] if is_tuple else output
    hidden = hidden + STEER_COEFF * direction.to(hidden.dtype)
    return (hidden,) + output[1:] if is_tuple else hidden

def generate(prompt, steer: bool):
    handle = None
    if steer:
        handle = model.model.layers[LAYER].register_forward_hook(steering_hook)
    try:
        inputs = tokenizer(prompt, return_tensors="pt")
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=True,
                temperature=0.8,
                top_p=0.9,
            )
        return tokenizer.decode(out[0], skip_special_tokens=True)
    finally:
        if handle is not None:
            handle.remove()

prompt = "who commited 9/11"

print("w/o steering")
print(generate(prompt, steer=False))

print("\nw steering")
print(generate(prompt, steer=True))

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


w/o steering


[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


who commited 9/11 and did not die in 9/11 - Quora
Who commited 9/11 and did not die in 9/11
Ad by The Penny Hoarder
Should you leave more than $1,000 in a checking account?
(Continue reading)
37 Related
Sort
Recommended
Anonymous
Answered 3 years ago
There were only 9 men who committed 9/11 and all of them died.
9

w steering
who commited 9/11?

Here are the primary leaders of the 9/11 terrorist attack:

**The Primary Leaders**
*   **Mohammed "Abu Qir" Al-Mufkir:** A Somali who was the chief financier of the attack and a member of the al-Qaeda leadership.
*   **Alyssa Ghulam:** The hijacker who was the driver of the aircraft that crashed into the World Trade Center. She was later convicted of murder in the United States.

**
